# Fill-in-the-blank: signal filtering

This worksheet uses fixed armband-style data with the same shape as your program: 100 frames at 50 Hz and 12 magnetic-field channels. Fill in each marked None value in order. The data setup and all project functions are already provided.

## Instructor flow

Ask learners to run one exercise at a time. Every checkpoint gives the expected shape. If a learner is stuck, point them to the project function named in the hint; do not reveal the next cell yet.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")

def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "calibration_pipeline" / "eflesh_calibration" / "knn.py").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from inside the project repository.")

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "calibration_pipeline"))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

from eflesh_calibration.knn import (
    FILTER_SAMPLES, RAW_WINDOW_SAMPLES, SAMPLE_HZ, WINDOW_SAMPLES,
    feature,
)
from teaching_samples import load_teaching_dataset

data = load_teaching_dataset()
time_s = data["time_s"]
baseline = data["baseline"]
captures = data["captures"]
labels = data["labels"]
channel_names = data["channel_names"]

print("captures:", captures.shape)
print("sample rate:", SAMPLE_HZ, "Hz")
print("filter frames:", FILTER_SAMPLES)

## Exercise 1 — choose data

Goal: select the first wrist_up capture and Sensor 0 X.

Target: raw_capture has shape (100, 12), and channel_name is S0x.

Hint: labels is an array. np.flatnonzero returns the positions where a condition is true.

In [ ]:
TARGET_LABEL = "wrist_up"

# TODO: replace each None.
capture_index = None
raw_capture = None
channel = None

channel_name = channel_names[channel] if channel is not None else None

assert capture_index is not None, "Choose the first matching capture index."
assert raw_capture is not None and raw_capture.shape == (100, 12)
assert channel_name == "S0x"
print("selected:", labels[capture_index], channel_name)

## Exercise 2 — show raw measurement and baseline

Goal: select one channel from the capture and its matching relaxed baseline value.

Hint: one channel is a column, so select all rows and one channel index.

In [ ]:
assert raw_capture is not None, "Complete Exercise 1 first."

# TODO: replace both None values.
raw_signal = None
channel_baseline = None

assert raw_signal.shape == time_s.shape
assert np.isscalar(channel_baseline)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(time_s, raw_signal, color="0.55", label="raw")
ax.axhline(channel_baseline, color="black", ls="--", label="baseline")
ax.axvspan(0.5, 1.5, color="tab:green", alpha=0.12, label="clean center")
ax.set(title=f"Raw capture: {labels[capture_index]}, {channel_name}", xlabel="time (s)", ylabel="magnetic field (µT)")
ax.legend()
plt.show()

## Exercise 3 — write the causal moving-average function

**Goal:** implement the filter yourself. Do not import it from the project.

A five-frame causal average makes one output from five consecutive input frames. With the sliding-window view below, the five samples for a channel are in the last axis.

**Targets:**

- a 100 by 12 input produces a 96 by 12 output;
- the small check produces 2, 3, 4, 5 in its first channel.

**Hints:** use NumPy sliding_window_view with the frame axis, then take the mean across the five-value axis.

In [ ]:
def moving_average(samples):
    samples = np.asarray(samples, dtype=float)

    # TODO: replace both None values. Do not change the function signature.
    neighborhoods = None
    filtered = None

    return filtered

# A tiny, known example checks your implementation before using armband data.
check_input = np.repeat(np.arange(8, dtype=float)[:, np.newaxis], 12, axis=1)
check_output = moving_average(check_input)

assert check_output is not None, "Return the filtered array."
assert check_output.shape == (4, 12), "Eight rows with a five-frame filter make four rows."
np.testing.assert_allclose(check_output[:, 0], [2.0, 3.0, 4.0, 5.0])
print("Moving-average checkpoint passed.")

## Exercise 4 — apply your filter

This cell has no plotting code. Its only job is to apply the function you wrote and align the output times.

**Target:** the filtered capture has shape (96, 12). The first filtered row aligns with input time index FILTER_SAMPLES minus one.

In [ ]:
assert raw_capture is not None, "Complete Exercise 1 first."

smoothed_capture = moving_average(raw_capture)
smoothed_time_s = time_s[FILTER_SAMPLES - 1 :]

assert smoothed_capture.shape == (96, 12)
assert smoothed_time_s.shape == (96,)
print("100 raw frames →", len(smoothed_capture), "filtered frames")

## Exercise 5 — visualize the result

The plotting logic is supplied. Run it after the moving-average checkpoint and compare the raw gray signal with your filtered blue signal.

Discuss: why is the blue line smoother, and why does it start later?

In [ ]:
filtered_signal = smoothed_capture[:, channel]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(time_s, raw_signal, color="0.75", label="raw")
ax.plot(
    smoothed_time_s,
    filtered_signal,
    color="tab:blue",
    lw=2,
    label="your five-frame causal average",
)
ax.axhline(channel_baseline, color="black", ls="--", label="baseline")
ax.set(
    title="Filtering used by the program",
    xlabel="time (s)",
    ylabel="magnetic field (µT)",
)
ax.legend()
plt.show()

## Exercise 6 — make the first real training window

The program starts its clean center at frame 25, then builds a 19-row raw window that becomes a 15-row filtered window.

Goals and targets:

- smoothed_start is 25
- raw_start is 21
- raw_window has shape (19, 12)
- smoothed_window has shape (15, 12)

Hint: a five-frame causal filter needs four earlier raw frames.

In [ ]:
CALIBRATION_EDGE_SAMPLES = SAMPLE_HZ // 2

# TODO: replace the four None values.
smoothed_start = None
raw_start = None
raw_window = None
smoothed_window = None

assert smoothed_start == 25
assert raw_start == 21
assert raw_window.shape == (RAW_WINDOW_SAMPLES, 12)
assert smoothed_window.shape == (WINDOW_SAMPLES, 12)
print("shapes:", raw_window.shape, "→", smoothed_window.shape)

## Exercise 7 — create the 24 features used by KNN

Goal: call feature with the raw window and the relaxed baseline.

Target: features has shape (24). The first 12 values are signed means; the final 12 are RMS values.

In [ ]:
# TODO: replace None.
features = None

assert features is not None and features.shape == (24,)
print("signed means:", np.round(features[:12], 2))
print("RMS values:  ", np.round(features[12:], 2))

## Done

You reproduced the actual path before KNN:

100-frame capture → 19-frame raw window → 15-frame filtered window → 24 features.

Open the KNN worksheet next. It uses the same prepared captures.